# 03 - data preprocessing (sample)

silence trim, normalize amplitude, pad/truncate to 3s on sample data.

In [ ]:
import pandas as pd
import numpy as np
import librosa
import sys
sys.path.append('../../')
from src.config.settings import SAMPLES, SAMPLE_RATE, CLIP_SAMPLES, TOP_DB


In [2]:
df = pd.read_csv(SAMPLES / "sample_labels.csv")
df.head()

,filepath,channel,emotion_code,emotion,intensity,statement,repetition,actor,gender,split
0,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,1,1,23,male,test
1,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,1,2,23,male,test
2,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,2,1,23,male,test
3,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,2,2,23,male,test
4,/mnt/d/career/projects/lightweight-speech-emot...,song,2,calm,normal,1,1,23,male,test


In [3]:
print(f"{len(df)} files to process")

416 files to process


In [4]:
import time
t0 = time.time()
for i, fp in enumerate(df['filepath']):
    y, sr = librosa.load(fp, sr=SAMPLE_RATE)
    yt, _ = librosa.effects.trim(y, top_db=TOP_DB)
    yn = yt / np.max(np.abs(yt) + 1e-9)
    if len(yn) < CLIP_SAMPLES:
        yn = np.pad(yn, (0, CLIP_SAMPLES - len(yn)))
    else:
        yn = yn[:CLIP_SAMPLES]
    if i == 0:
        print(f"sample shape: {yn.shape}, sr={sr}")
t1 = time.time()
print(f"preprocessed {len(df)} clips in {t1-t0:.1f}s")

sample shape: (144000,), sr=48000
preprocessed 416 clips in 39.1s


In [5]:
print("preprocessing verification:")
y,_ = librosa.load(df['filepath'].iloc[0], sr=SAMPLE_RATE)
yt,_ = librosa.effects.trim(y, top_db=TOP_DB)
print(f"  original: {len(y)} samples ({len(y)/SAMPLE_RATE:.2f}s)")
print(f"  trimmed:  {len(yt)} samples ({len(yt)/SAMPLE_RATE:.2f}s)")
print(f"  norm max: {np.max(np.abs(yt/np.max(np.abs(yt)+1e-9))):.3f}")

preprocessing verification:
  original: 211411 samples (4.40s)
  trimmed:  113152 samples (2.36s)
  norm max: 1.000


preprocessing pipeline verified on sample data.